# multi_4h_jma_atr_021

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (184).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 10 |
| Detected functions | resample_1hr_custom, __init__, compute_atr, compute_jma_and_signals, compute_atr_max, py_backtest_trades, calc_pnl, backtest_trades |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Data/Crypto_2025/Crypto_T5_Sept_2025.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2022-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['SOLUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

### **INDICATOR CODE**

In [ ]:
import pandas as pd
import numpy as np

# ===============================
# ATR Calculator
# ===============================
class ATRIndicator:
    """Average True Range (ATR) Calculator"""
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high = df[self.high_col]
        low = df[self.low_col]
        close = df[self.close_col]

        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low - close.shift(1)).abs()
        ], axis=1).max(axis=1)

        atr = tr.rolling(window=period, min_periods=1).mean()
        return atr.shift(shift)  # shift to prevent lookahead

# ===============================
# JMA + Signals (manual loop)
# ===============================
def compute_jma_and_signals(df,
                            smooth_length=14,
                            normalization_period=36,
                            upper_threshold=1.75,
                            lower_threshold=-1.25,
                            rs_period=14,
                            rs_percentile=10):
    df = df.copy()
    price = df['close']

    # --- JMA calculation (manual loop) ---
    alpha = 2.0 / (smooth_length + 1)
    jma_values = [price.iloc[0]]
    for p in price.iloc[1:]:
        jma_values.append(jma_values[-1] + (p - jma_values[-1]) * alpha)
    df['JMA'] = jma_values

    # --- Normalization ---
    df['Normalized_JMA'] = (df['JMA'] - df['JMA'].rolling(normalization_period).mean()) / \
                            df['JMA'].rolling(normalization_period).std()

    # --- Rogers-Satchell Volatility ---
    o = df['open'].replace(0, np.nan)
    h = df['high']
    l = df['low']
    c = df['close'].replace(0, np.nan)

    log_hc = np.log(h / c)
    log_ho = np.log(h / o)
    log_lc = np.log(l / c)
    log_lo = np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6

    # --- RS Volatility Filter ---
    df['RS_Threshold'] = df['RS_Volatility'].rolling(normalization_period).quantile(rs_percentile / 100.0)

    # --- Signals ---
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 2

    return df

# ===============================
# ATR_max computation helper
# ===============================
def compute_atr_max(group):
    return pd.concat([
        group['ATR_14'].rolling(20).max() * 0.2,
        group['ATR_5'].rolling(20).max() * 0.2
    ], axis=1).max(axis=1)

# ===============================
# Full Data Preparation
# ===============================
# Assuming 'data' DataFrame exists with required columns
data['Date'] = pd.to_datetime(data['Date'])

# Apply JMA + Signals per ticker
jma_results = []
for ticker, group in data.groupby('Ticker'):
    jma_group = compute_jma_and_signals(group)
    jma_results.append(jma_group)
data = pd.concat(jma_results, ignore_index=True)

# Initialize ATR calculator
atr_calc = ATRIndicator()

# Compute ATRs per ticker with shift=1
for period in [14, 5]:
    col_name = f'ATR_{period}'
    atr_results = []
    for ticker, group in data.groupby('Ticker'):
        atr_vals = atr_calc.compute_atr(group, period=period, shift=1)
        atr_results.append(atr_vals)
    data[col_name] = pd.concat(atr_results, ignore_index=True)

# Compute ATR_max
atr_max_results = []
for ticker, group in data.groupby('Ticker'):
    atr_max_vals = compute_atr_max(group)
    atr_max_results.append(atr_max_vals)
data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)

# Shift signals for next-bar execution
data['Signal'] = data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(np.int8)

# Ready for backtesting
data

In [ ]:
data = data[[
    "Ticker", "Date", "open", "high", "low", "close", "Signal", "ATR_max"
]]
data

# **BACKTESTING CODE**

In [ ]:
import numpy as np
import pandas as pd

def py_backtest_trades(
    data: np.ndarray,
    ticker: str,
    capital: float,
    fee_rate: float,
    slippage_rate: float,
    entry_sl_pct: float,   # constant entry SL for entry candle
    max_daily_dd_pct: float,  # daily drawdown limit
):
    """
    Refined Backtest Engine with Your Exact Logic

    Logic:
    - Entry Candle: Only entry SL check
    - Candle 2+:
      1. First check trailing SL
      2. Then check direction (open & close in our favor)
      3. Exit at close price (with slippage) if direction unfavorable
    """

    def calc_pnl(entry_price, exit_price, qty, direction):
        """Helper to compute PnL with fees."""
        pnl = (exit_price - entry_price) * qty * direction
        fee = fee_rate * (entry_price + exit_price) * qty
        return pnl - fee

    n = data.shape[0]
    trades, cumulative_pnl = [], 0.0

    # --- Trade State ---
    position_open = False
    entry_price = sl_price = quantity = 0.0
    entry_time = last_exit_time = None
    direction = 0  # 1 = long, -1 = short

    # --- Previous Candle (for trailing SL) ---
    prev_high = prev_low = prev_close = prev_atr_max = 0.0

    # --- Daily Risk Control ---
    daily_pnl = 0.0
    last_day = None
    max_daily_loss = -capital * max_daily_dd_pct
    trading_blocked_today = False

    for i in range(n):
        time, open_, high, low, close, signal, atr_max = (
            pd.to_datetime(data[i, 0]),
            float(data[i, 1]), float(data[i, 2]),
            float(data[i, 3]), float(data[i, 4]), int(data[i, 5]),
            float(data[i, 6])
        )
        day = time.date()

        # Reset daily PnL + trading block at new day
        if last_day is None or day != last_day:
            daily_pnl = 0.0
            trading_blocked_today = False
            last_day = day

        # ======================
        #  ENTRY LOGIC
        # ======================
        if (
            not position_open
            and not trading_blocked_today
            and time != last_exit_time
            and signal in (1, 2)
        ):
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            quantity = capital / entry_price
            entry_time, position_open = time, True

            # Flash-crash protection (constant entry SL)
            sl_price = entry_price * (1 - entry_sl_pct if direction == 1 else 1 + entry_sl_pct)

            # ENTRY CANDLE: Only check entry SL
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl
                daily_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, "Entry SL", cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time

                # Block trading if daily loss breached
                if daily_pnl <= max_daily_loss:
                    trading_blocked_today = True
            continue

        # ======================
        #  POSITION MANAGEMENT (CANDLE 2+)
        # ======================
        if position_open and time != entry_time:
            exit_triggered = False
            exit_price = None
            exit_reason = None

            # STEP 1: Check Trailing SL FIRST (Priority)
            # Update trailing SL using previous candle
            if i > 0:
                if direction == 1:
                    sl_price = max(sl_price, prev_close - prev_atr_max)
                else:
                    sl_price = min(sl_price, prev_close + prev_atr_max)

            # Check if trailing SL hit
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                exit_reason = "Trailing SL"
                exit_triggered = True

            # STEP 2: Check Direction (Candle Color/Momentum) - Only if Trailing SL not hit
            else:
                # Check candle direction/color
                unfavorable_candle = False

                if direction == 1:  # LONG position
                    # Want green candle (open < close)
                    if open_ >= close:  # Red candle is unfavorable
                        unfavorable_candle = True
                else:  # SHORT position
                    # Want red candle (close < open)
                    if close >= open_:  # Green candle is unfavorable
                        unfavorable_candle = True

                if unfavorable_candle:
                    # Exit at CLOSE price with slippage
                    exit_price = close * (1 - slippage_rate if direction == 1 else 1 + slippage_rate)
                    exit_reason = "Not in direction (candle color)"
                    exit_triggered = True

            # Execute exit if any condition triggered
            if exit_triggered:
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl
                daily_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, exit_reason, cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time

                # Block trading if daily loss breached
                if daily_pnl <= max_daily_loss:
                    trading_blocked_today = True

        # Store previous candle (for trailing SL reference)
        prev_high, prev_low, prev_close, prev_atr_max = high, low, close, atr_max

    return trades

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from typing import List, Tuple

def backtest_trades(
    data: pd.DataFrame,
    entry_sl_pct: float,
    initial_capital: float,
    fee_rate: float,
    slippage_rate: float,
    max_daily_dd_pct: float,
) -> pd.DataFrame:
    """
    Wrapper for running py_backtest_trades_v3 on multiple tickers.
    Converts DataFrame -> numpy, runs backtest, returns DataFrame of trades.
    """

    # Ensure datetime format
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()

    # Prepare data arrays for each ticker
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][[
            "Date", "open", "high", "low", "close", "Signal", "ATR_max"
        ]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades: List[Tuple] = []

    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = py_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            entry_sl_pct,
            max_daily_dd_pct
        )
        all_trades.extend(trades)

    # Convert trades into DataFrame
    tradebook = pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])
    return tradebook


# === Example Usage ===
tradebook = backtest_trades(
    data,
    entry_sl_pct=0.02,
    initial_capital=100000,
    fee_rate=0.0005,
    slippage_rate=0.0005,
    max_daily_dd_pct= 0.03,
)

# Save & inspect
tradebook.to_csv("/content/drive/MyDrive/tradebook_SOLUSDT_2025.csv", index=False)
tradebook